In [4]:
"""
Add Uniform CD Number + One-Year Backward Shift
================================================
Combines two steps that used to be separate:

1. Derives `cd_number_uniform` from `cd_number` / `cd_geoid` so every year
   uses the same CD identifier scheme (1984-2012 from cd_number, 2013+
   from the last two digits of cd_geoid). At-large districts stay as '0'.

2. Shifts every row's `year` backward by 1 year to correct for the data
   storage glitch: the CD geometry stored under year Y actually reflects
   the district boundaries that were in force during session Y-1.

   After shifting:
     - original 1984  -> 1983
     - original 2012  -> 2011
     - original 2025  -> 2024

   IMPORTANT: the uniform CD extraction runs on the ORIGINAL year label,
   because the `year <= 2012` branch in the extraction logic was written
   against the original column conventions. Shifting happens after.

Input : matches.csv (the raw output of the matcher)
Output: matches_with_uniform_cd_shifted.csv
"""

import pandas as pd


def extract_uniform_cd_number(row):
    """Extract a uniform CD number from cd_number / cd_geoid.

    NOTE: `row['year']` here is the ORIGINAL (unshifted) year. This matters
    because the storage convention — cd_number for 1984-2012, cd_geoid-
    derived for 2013+ — is tied to the original year labels, not the
    corrected ones.
    """
    year = row["year"]
    cd_number = str(row["cd_number"]) if pd.notna(row["cd_number"]) else ""
    cd_geoid = str(row["cd_geoid"]) if pd.notna(row["cd_geoid"]) else ""

    if year <= 2012:
        # 1984-2012: cd_number already contains the district number.
        # Strip leading zeros; at-large stays as '0'.
        return cd_number.lstrip("0") if cd_number.lstrip("0") else "0"
    else:
        # 2013+: cd_geoid is state FIPS + CD number. Last two digits are CD.
        if len(cd_geoid) >= 2:
            cd_from_geoid = cd_geoid[-2:].lstrip("0")
            return cd_from_geoid if cd_from_geoid else "0"
        # Fallback when cd_geoid is missing.
        return cd_number.lstrip("0") if cd_number.lstrip("0") else "0"


def main(
    input_file: str = "matches.csv",
    output_file: str = "matches_with_uniform_cd_shifted.csv",
    year_shift: int = -1,
):
    print("=" * 80)
    print("UNIFORM CD NUMBER + ONE-YEAR BACKWARD SHIFT")
    print("=" * 80)

    print(f"\nLoading {input_file}...")
    df = pd.read_csv(input_file, low_memory=False)
    print(f"  {len(df):,} rows, original year range: "
          f"{df['year'].min()} - {df['year'].max()}")

    # --- Step 1: derive uniform CD number from the ORIGINAL year -----------
    print("\nStep 1: deriving cd_number_uniform from original year labels...")
    df["cd_number_uniform"] = df.apply(extract_uniform_cd_number, axis=1)
    print("  Done.")

    # --- Step 2: preserve original year, then shift ------------------------
    # Keeping the original year as a new column is useful for debugging /
    # auditing: if anything looks odd downstream, you can always go back to
    # the raw label.
    print(f"\nStep 2: shifting year by {year_shift} (glitch correction)...")
    df["year_original"] = df["year"]
    df["year"] = df["year"] + year_shift
    print(f"  Corrected year range: {df['year'].min()} - {df['year'].max()}")

    # --- Reorder columns ----------------------------------------------------
    # Put cd_number_uniform right after cd_geoid, and year_original right
    # after year, so the CSV reads naturally.
    cols = df.columns.tolist()
    cols.remove("cd_number_uniform")
    cols.remove("year_original")

    if "cd_geoid" in cols:
        cols.insert(cols.index("cd_geoid") + 1, "cd_number_uniform")
    else:
        cols.append("cd_number_uniform")

    if "year" in cols:
        cols.insert(cols.index("year") + 1, "year_original")
    else:
        cols.append("year_original")

    df = df[cols]

    # --- Save ---------------------------------------------------------------
    print(f"\nSaving to {output_file}...")
    df.to_csv(output_file, index=False)
    print("  Saved.")

    # --- Summary ------------------------------------------------------------
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Total rows                : {len(df):,}")
    print(f"Original years (input)    : "
          f"{df['year_original'].min()} - {df['year_original'].max()}")
    print(f"Corrected years (output)  : "
          f"{df['year'].min()} - {df['year'].max()}")
    print(f"Unique uniform CD numbers : {df['cd_number_uniform'].nunique()}")

    print("\nSample rows showing the shift (one per original-year milestone):")
    milestones = [1984, 2000, 2012, 2013, 2020, 2025]
    for y in milestones:
        hit = df[df["year_original"] == y]
        if len(hit) == 0:
            continue
        sample = hit[
            ["year_original", "year", "state_name", "cd_number",
             "cd_geoid", "cd_number_uniform"]
        ].head(1)
        print(f"\nOriginal year {y}:")
        print(sample.to_string(index=False))

    print("\nDone. Next step: run your redistricting analysis on "
          f"{output_file}, using `year` (now corrected) as the time key.")

    return df


if __name__ == "__main__":
    # Update these paths as needed.
    INPUT_FILE = "matches_state_filled.csv"
    OUTPUT_FILE = "matches_with_uniform_cd_shifted1.csv"

    main(input_file=INPUT_FILE, output_file=OUTPUT_FILE, year_shift=-1)

UNIFORM CD NUMBER + ONE-YEAR BACKWARD SHIFT

Loading matches_state_filled.csv...
  150,539 rows, original year range: 1984 - 2025

Step 1: deriving cd_number_uniform from original year labels...
  Done.

Step 2: shifting year by -1 (glitch correction)...
  Corrected year range: 1983 - 2024

Saving to matches_with_uniform_cd_shifted1.csv...
  Saved.

SUMMARY
Total rows                : 150,539
Original years (input)    : 1984 - 2025
Corrected years (output)  : 1983 - 2024
Unique uniform CD numbers : 110

Sample rows showing the shift (one per original-year milestone):

Original year 1984:
 year_original  year state_name  cd_number cd_geoid cd_number_uniform
          1984  1983    Alabama        1.0      NaN               1.0

Original year 2000:
 year_original  year state_name  cd_number cd_geoid cd_number_uniform
          2000  1999    Alabama        1.0      NaN               1.0

Original year 2012:
 year_original  year state_name  cd_number cd_geoid cd_number_uniform
          201

In [5]:
"""
Congressional District Redistricting Analysis (Year-Shifted Version)
=====================================================================
Analyzes congressional district changes over time to identify redistricting
events, using the year-shifted matches file produced by
`add_uniform_cd_and_shift_year.py`.

Input : matches file with columns
          - year              (corrected year after -1 shift)
          - year_original     (pre-shift year; optional, carried through)
          - cd_number_uniform (consistent CD identifier across all years)
          - state_name, county_name, county_fips, pct_cd_in_county
Output: CSV with State, District, Year, Relevant Counties, District Same?

Redistricting detection:
  - For each (state, district), compare county composition year-over-year.
  - total_change = sum of
        + removed counties' previous %
        + added counties' current %
        + abs(pct change) for counties in both years
  - If total_change > THRESHOLD, flag as redistricting (District Same? = 1).

After the one-year glitch correction, post-census redistricting spikes
should appear at 2011 and 2021 (instead of 2012 and 2022), which lines up
with the real redistricting cycles following the 2010 and 2020 censuses.
"""

import pandas as pd
import numpy as np


# ---------------------------------------------------------------------------
# Configuration — tweak these in one place.
# ---------------------------------------------------------------------------
INPUT_FILE = "/Users/adrianneli/matches_with_uniform_cd_shifted1.csv"
OUTPUT_FILE = "redistricting_analysis_shifted1.csv"
SUMMARY_FILE = "redistricting_summary_shifted1.txt"

# Percentage threshold for flagging redistricting. Your original code used
# 20.0; the FIXED draft used 10.0 in one place. 20.0 is the safer default
# for catching real redistricting cycles vs. noise — flip this to 10.0 if
# you want a stricter definition.
REDISTRICTING_THRESHOLD = 20.0


# ---------------------------------------------------------------------------
# Data loading & preparation
# ---------------------------------------------------------------------------
def load_and_prepare_data(matches_file):
    """Load the shifted matches CSV and normalize types."""
    print("Loading matches data...")
    matches = pd.read_csv(matches_file, low_memory=False)

    if "cd_number_uniform" in matches.columns:
        cd_col = "cd_number_uniform"
        print("Using 'cd_number_uniform' column for analysis")
    else:
        cd_col = "cd_number"
        print(
            "[warn] 'cd_number_uniform' not found — falling back to "
            "'cd_number'. Run add_uniform_cd_and_shift_year.py first for "
            "consistent cross-year district identification."
        )

    # Sanity-check that the year shift has already been applied. If the
    # file was shifted, it will either have `year_original` as a column OR
    # its max year will be 2024 (not 2025). If neither is true, warn loudly.
    has_year_original = "year_original" in matches.columns
    max_year = matches["year"].max()
    if not has_year_original and max_year >= 2025:
        print(
            "[warn] This file doesn't look year-shifted (max year is "
            f"{max_year} and 'year_original' is missing). If you intended "
            "to use shifted data, run add_uniform_cd_and_shift_year.py "
            "first."
        )
    elif has_year_original:
        print(
            f"Year shift confirmed: original {matches['year_original'].min()}"
            f"-{matches['year_original'].max()} -> corrected "
            f"{matches['year'].min()}-{matches['year'].max()}"
        )

    # Drop rows missing state or CD.
    initial_rows = len(matches)
    matches = matches[
        matches["state_name"].notna() & matches[cd_col].notna()
    ].copy()
    removed = initial_rows - len(matches)
    if removed:
        print(f"Removed {removed} rows with missing state_name or {cd_col}")

    # Ensure we always have cd_number_uniform, even if we fell back above.
    if "cd_number_uniform" not in matches.columns:
        matches["cd_number_uniform"] = matches[cd_col]

    # Normalize CD number type: string -> strip -> drop trailing ".0" ->
    # numeric. This keeps "1" and "1.0" from being treated as different
    # districts (the bug the FIXED draft addressed).
    matches["cd_number_uniform"] = (
        matches["cd_number_uniform"].astype(str).str.strip()
    )
    matches["cd_number_uniform"] = matches["cd_number_uniform"].str.replace(
        r"\.0$", "", regex=True
    )
    try:
        matches["cd_number_uniform"] = pd.to_numeric(
            matches["cd_number_uniform"], errors="coerce"
        )
        matches["cd_number_uniform"] = (
            matches["cd_number_uniform"].fillna(0).astype(int)
        )
        print("Converted cd_number_uniform to integer type")
    except Exception as e:
        print(f"Keeping CD numbers as string: {e}")

    print(f"\nData loaded: {len(matches):,} rows")
    print(f"Year range (corrected): {matches['year'].min()} - {matches['year'].max()}")
    print(f"States: {matches['state_name'].nunique()}")
    print(f"Unique districts: {matches['cd_number_uniform'].nunique()}")
    print(f"CD number dtype: {matches['cd_number_uniform'].dtype}")

    print("\nSample (first 10 unique state-district-year combos):")
    sample = (
        matches[["state_name", "cd_number_uniform", "year"]]
        .drop_duplicates()
        .head(10)
    )
    print(sample.to_string(index=False))

    return matches


# ---------------------------------------------------------------------------
# Aggregation & change detection
# ---------------------------------------------------------------------------
def format_county_with_percentage(county_name, pct):
    return f"{county_name} ({pct:.2f}%)"


def aggregate_by_district_year(matches):
    """Collapse to one row per (state, district, corrected year) with the
    full list of counties and their percentages."""
    print("\nAggregating by state-district-year (using corrected year)...")

    grouped = (
        matches.groupby(["state_name", "cd_number_uniform", "year"])
        .agg(
            {
                "county_name": list,
                "county_fips": list,
                "pct_cd_in_county": list,
            }
        )
        .reset_index()
    )
    grouped = grouped.sort_values(
        ["state_name", "cd_number_uniform", "year"]
    ).reset_index(drop=True)

    print(f"Total state-district-year combinations: {len(grouped):,}")
    print("\nFirst 5 rows of aggregated data:")
    print(grouped[["state_name", "cd_number_uniform", "year"]].head())

    return grouped


def calculate_change_percentage(current_row, previous_row):
    """Sum of removed %, added %, and retained-but-shifted % between years."""
    curr_counties = set(current_row["county_fips"])
    prev_counties = set(previous_row["county_fips"])

    curr_pct = dict(
        zip(current_row["county_fips"], current_row["pct_cd_in_county"])
    )
    prev_pct = dict(
        zip(previous_row["county_fips"], previous_row["pct_cd_in_county"])
    )

    total = 0.0
    # Removed counties contribute their full previous share.
    for c in prev_counties - curr_counties:
        total += prev_pct[c]
    # Added counties contribute their full current share.
    for c in curr_counties - prev_counties:
        total += curr_pct[c]
    # Retained counties contribute their absolute shift.
    for c in curr_counties & prev_counties:
        total += abs(curr_pct[c] - prev_pct[c])

    return total


def detect_redistricting(current_row, previous_row, threshold):
    """Return 1 if redistricting, 0 if stable, None for baseline year."""
    if previous_row is None:
        return None
    return 1 if calculate_change_percentage(current_row, previous_row) > threshold else 0


def create_redistricting_analysis(grouped, threshold):
    """Walk each (state, district) timeline and flag changes."""
    print(f"\nDetecting redistricting events (threshold: {threshold}%)...")

    results = []
    for (state, district), group in grouped.groupby(
        ["state_name", "cd_number_uniform"]
    ):
        group = group.sort_values("year").reset_index(drop=True)

        for idx, row in group.iterrows():
            if idx == 0:
                district_same = None  # baseline year
            else:
                prev_row = group.iloc[idx - 1]
                district_same = detect_redistricting(row, prev_row, threshold)

            counties_fmt = [
                format_county_with_percentage(n, p)
                for n, p in zip(row["county_name"], row["pct_cd_in_county"])
            ]
            # Alphabetical sort up front — saves a downstream sort script.
            counties_fmt = sorted(counties_fmt, key=lambda s: s.lower())

            results.append(
                {
                    "State": state,
                    "District": district,
                    "Year": int(row["year"]),
                    "Relevant Counties": ", ".join(counties_fmt),
                    "District Same?": district_same,
                }
            )

    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values(
        ["State", "District", "Year"]
    ).reset_index(drop=True)

    total = len(result_df)
    baseline = result_df["District Same?"].isna().sum()
    stable = (result_df["District Same?"] == 0).sum()
    redistricted = (result_df["District Same?"] == 1).sum()

    print(f"\nAnalysis complete.")
    print(f"  Total records       : {total:,}")
    print(f"  Baseline years      : {baseline:,}")
    print(f"  Stable years        : {stable:,}")
    print(f"  Redistricting events: {redistricted:,}")

    return result_df


# ---------------------------------------------------------------------------
# Summary report
# ---------------------------------------------------------------------------
def generate_summary_report(result_df, output_file, threshold):
    print(f"\nGenerating summary report...")
    with open(output_file, "w") as f:
        f.write("=" * 80 + "\n")
        f.write("CONGRESSIONAL DISTRICT REDISTRICTING ANALYSIS (YEAR-SHIFTED)\n")
        f.write("=" * 80 + "\n\n")

        f.write("METHODOLOGY:\n")
        f.write("-" * 80 + "\n")
        f.write(
            "- Year labels corrected by -1 to fix the data-storage glitch in\n"
            "  the original matches file (geometry stored under year Y\n"
            "  actually reflects the session that held in year Y-1).\n"
            f"- cd_number_uniform gives a consistent district ID across the\n"
            f"  entire time series (1984-2012 from cd_number, 2013+ from\n"
            f"  the last 2 digits of cd_geoid).\n"
            f"- Threshold: >{threshold}% change in district composition\n"
            f"  triggers a redistricting flag.\n"
            f"- Change = removed-county % + added-county % + |retained %|.\n"
            f"- District Same? codes: 0 stable, 1 redistricted, NaN baseline.\n\n"
        )

        f.write("OVERALL STATISTICS:\n")
        f.write("-" * 80 + "\n")
        f.write(f"Total records: {len(result_df):,}\n")
        f.write(
            f"Time period (corrected): {result_df['Year'].min()} - "
            f"{result_df['Year'].max()}\n"
        )
        f.write(f"States analyzed: {result_df['State'].nunique()}\n")
        f.write(
            f"Unique state-district combinations: "
            f"{result_df[['State', 'District']].drop_duplicates().shape[0]}\n\n"
        )

        baseline = result_df["District Same?"].isna().sum()
        stable = (result_df["District Same?"] == 0).sum()
        redistricted = (result_df["District Same?"] == 1).sum()
        total = len(result_df)
        f.write(
            f"Baseline years (no comparison): {baseline:,} "
            f"({baseline/total*100:.1f}%)\n"
        )
        f.write(
            f"Stable years: {stable:,} ({stable/total*100:.1f}%)\n"
        )
        f.write(
            f"Redistricting events: {redistricted:,} "
            f"({redistricted/total*100:.1f}%)\n\n"
        )

        f.write("REDISTRICTING EVENTS BY YEAR (Top 15):\n")
        f.write("-" * 80 + "\n")
        by_year = (
            result_df[result_df["District Same?"] == 1]
            .groupby("Year")
            .size()
            .sort_values(ascending=False)
        )
        for year, count in by_year.head(15).items():
            f.write(f"  {year}: {count:>4} districts\n")

        f.write(
            "\nNOTE: After the -1 year shift, post-census redistricting\n"
            "spikes should fall on 1991, 2001, 2011, and 2021 — the years\n"
            "immediately following the decennial censuses.\n\n"
        )

        f.write("STATES WITH MOST REDISTRICTING ACTIVITY (Top 20):\n")
        f.write("-" * 80 + "\n")
        by_state = (
            result_df[result_df["District Same?"] == 1]
            .groupby("State")
            .size()
            .sort_values(ascending=False)
        )
        for state, count in by_state.head(20).items():
            f.write(f"  {state:<25} {count:>4} events\n")

    print(f"Summary report saved to: {output_file}")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
def main():
    print("=" * 80)
    print("CONGRESSIONAL DISTRICT REDISTRICTING ANALYSIS (YEAR-SHIFTED)")
    print("=" * 80)

    matches = load_and_prepare_data(INPUT_FILE)
    grouped = aggregate_by_district_year(matches)
    result_df = create_redistricting_analysis(
        grouped, threshold=REDISTRICTING_THRESHOLD
    )

    print(f"\nSaving results to: {OUTPUT_FILE}")
    result_df.to_csv(OUTPUT_FILE, index=False)

    generate_summary_report(result_df, SUMMARY_FILE, REDISTRICTING_THRESHOLD)

    print("\n" + "=" * 80)
    print("SAMPLE RESULTS (first 20 rows):")
    print("=" * 80)
    sample = result_df.head(20).copy()
    sample["Relevant Counties"] = sample["Relevant Counties"].apply(
        lambda x: x[:60] + "..." if len(x) > 60 else x
    )
    print(sample.to_string(index=False))

    print("\n" + "=" * 80)
    print("DONE")
    print("=" * 80)
    print(f"\nOutput files:")
    print(f"  1. {OUTPUT_FILE}")
    print(f"  2. {SUMMARY_FILE}")

    return result_df


if __name__ == "__main__":
    result_df = main()

CONGRESSIONAL DISTRICT REDISTRICTING ANALYSIS (YEAR-SHIFTED)
Loading matches data...
Using 'cd_number_uniform' column for analysis
Year shift confirmed: original 1984-2025 -> corrected 1983-2024
Converted cd_number_uniform to integer type

Data loaded: 150,539 rows
Year range (corrected): 1983 - 2024
States: 56
Unique districts: 55
CD number dtype: int64

Sample (first 10 unique state-district-year combos):
state_name  cd_number_uniform  year
   Alabama                  1  1983
   Alabama                  2  1983
   Alabama                  3  1983
   Alabama                  4  1983
   Alabama                  5  1983
   Alabama                  6  1983
   Alabama                  7  1983
    Alaska                  0  1983
   Arizona                  1  1983
   Arizona                  2  1983

Aggregating by state-district-year (using corrected year)...
Total state-district-year combinations: 18,396

First 5 rows of aggregated data:
  state_name  cd_number_uniform  year
0    Alabama

In [3]:
"""
Backfill state_name in matches.csv
===================================
The matcher's pre-2013 pipeline reads UCLA Congressional District shapefiles
that don't carry state info in any column the matcher's `_standardize_columns`
was looking for. As a result, ~94% of rows from 1984-2012 end up with an
empty `state_name`, which propagates to the redistricting analysis (where
those rows then get dropped because they're missing state).

This patch reads the existing matches.csv, derives state from the first two
digits of `county_fips` (which IS fully populated for every row), and writes
a fixed version. No re-running the matcher required.

Before:  ~97,600 rows with missing state_name
After:   ~0 rows missing (plus a handful of territory rows that stay blank
         if you prefer to exclude them — see the TERRITORY_POLICY flag).
"""

import pandas as pd


# 50 states + DC — these are the rows we definitely want filled.
STATE_FIPS_MAP = {
    "01": "Alabama", "02": "Alaska", "04": "Arizona", "05": "Arkansas",
    "06": "California", "08": "Colorado", "09": "Connecticut", "10": "Delaware",
    "11": "District of Columbia", "12": "Florida", "13": "Georgia", "15": "Hawaii",
    "16": "Idaho", "17": "Illinois", "18": "Indiana", "19": "Iowa",
    "20": "Kansas", "21": "Kentucky", "22": "Louisiana", "23": "Maine",
    "24": "Maryland", "25": "Massachusetts", "26": "Michigan", "27": "Minnesota",
    "28": "Mississippi", "29": "Missouri", "30": "Montana", "31": "Nebraska",
    "32": "Nevada", "33": "New Hampshire", "34": "New Jersey", "35": "New Mexico",
    "36": "New York", "37": "North Carolina", "38": "North Dakota", "39": "Ohio",
    "40": "Oklahoma", "41": "Oregon", "42": "Pennsylvania", "44": "Rhode Island",
    "45": "South Carolina", "46": "South Dakota", "47": "Tennessee", "48": "Texas",
    "49": "Utah", "50": "Vermont", "51": "Virginia", "53": "Washington",
    "54": "West Virginia", "55": "Wisconsin", "56": "Wyoming",
}

# US territories. Flip TERRITORY_POLICY to control what happens to them.
TERRITORY_FIPS_MAP = {
    "60": "American Samoa",
    "66": "Guam",
    "69": "Northern Mariana Islands",
    "72": "Puerto Rico",
    "78": "US Virgin Islands",
}

# 'fill' labels territories with their name (kept in output, no blanks).
# 'drop' leaves them NaN so downstream analysis drops them naturally.
TERRITORY_POLICY = "fill"


def backfill_state_name(
    input_file: str = "matches.csv",
    output_file: str = "matches_state_filled.csv",
):
    print("=" * 78)
    print("BACKFILL state_name FROM county_fips")
    print("=" * 78)

    print(f"\nLoading {input_file}...")
    df = pd.read_csv(input_file, low_memory=False)
    print(f"  {len(df):,} rows loaded")

    # Derive a clean 2-char state FIPS from county_fips. county_fips can come
    # through as float (1003.0), int (1003), or string ('01003' / '1003'),
    # so normalize aggressively.
    fips_str = (
        df["county_fips"]
        .astype(str)
        .str.split(".").str[0]            # strip any '.0' from float coercion
        .str.replace(r"\D", "", regex=True)  # drop any stray non-digits
        .str.zfill(5)                     # zero-pad to 5 digits
    )
    state_fips_2 = fips_str.str[:2]

    # Build the combined FIPS -> state name mapping per policy.
    full_map = dict(STATE_FIPS_MAP)
    if TERRITORY_POLICY == "fill":
        full_map.update(TERRITORY_FIPS_MAP)

    derived_state_name = state_fips_2.map(full_map)

    # Before/after audit.
    missing_before = df["state_name"].isna().sum()
    print(f"\n  Before: {missing_before:,} rows with missing state_name")

    # Only fill where currently missing; don't overwrite rows that already
    # have a real state name from the matcher.
    needs_fill = df["state_name"].isna()
    df.loc[needs_fill, "state_name"] = derived_state_name[needs_fill]

    # Also fill state_fips column if present but blank, since the uniform-CD
    # step downstream sometimes needs it.
    if "state_fips" not in df.columns:
        df["state_fips"] = state_fips_2
    else:
        needs_fips_fill = df["state_fips"].isna() | (df["state_fips"].astype(str) == "")
        df.loc[needs_fips_fill, "state_fips"] = state_fips_2[needs_fips_fill]

    missing_after = df["state_name"].isna().sum()
    filled = missing_before - missing_after
    print(f"  Filled:  {filled:,} rows")
    print(f"  After:  {missing_after:,} rows still missing")

    if missing_after:
        leftover_fips = (
            state_fips_2[df["state_name"].isna()]
            .value_counts()
            .head(10)
        )
        print("\n  Unresolved FIPS prefixes (probably territories if "
              "TERRITORY_POLICY='drop'):")
        for fips, n in leftover_fips.items():
            name = TERRITORY_FIPS_MAP.get(fips, "unknown")
            print(f"    {fips} ({name}): {n:,} rows")

    # Quick per-year check so user can see coverage recovered.
    print("\n  Missing state_name by year (after fill):")
    by_year = df.groupby("year").apply(
        lambda x: (x["state_name"].isna().sum(), len(x))
    )
    for year, (miss, total) in by_year.items():
        marker = "" if miss == 0 else " <-- still missing"
        print(f"    {year}: {miss:>4} / {total:>5}{marker}")

    print(f"\nSaving to {output_file}...")
    df.to_csv(output_file, index=False)
    print("  Saved.")

    print("\n" + "=" * 78)
    print("DONE")
    print("=" * 78)
    print(f"\nNext: feed {output_file} into add_uniform_cd_and_shift_year.py\n"
          f"(set INPUT_FILE at the top of that script accordingly).")

    return df


if __name__ == "__main__":
    backfill_state_name(
        input_file="/Users/adrianneli/data/results/matches.csv",
        output_file="matches_state_filled.csv",
    )

BACKFILL state_name FROM county_fips

Loading /Users/adrianneli/data/results/matches.csv...
  150,539 rows loaded

  Before: 97,622 rows with missing state_name
  Filled:  97,622 rows
  After:  0 rows still missing

  Missing state_name by year (after fill):
    1984:    0 /  3498
    1985:    0 /  3503
    1986:    0 /  3508
    1987:    0 /  3508
    1988:    0 /  3521
    1989:    0 /  3506
    1990:    0 /  3519
    1991:    0 /  3509
    1992:    0 /  3504
    1993:    0 /  3606
    1994:    0 /  3611
    1995:    0 /  3586
    1996:    0 /  3585
    1997:    0 /  3568
    1998:    0 /  3556
    1999:    0 /  3544
    2000:    0 /  3532
    2001:    0 /  3533
    2002:    0 /  3533
    2003:    0 /  3616
    2004:    0 /  3616
    2005:    0 /  3626
    2006:    0 /  3626
    2007:    0 /  3607
    2008:    0 /  3607
    2009:    0 /  3607
    2010:    0 /  3607
    2011:    0 /  3607
    2012:    0 /  3607
    2013:    0 /  3681
    2014:    0 /  3680
    2015:    0 /  3680
    2